In [2]:
import sys, torch
print("Colab ortamında mıyız?:", 'google.colab' in sys.modules)
print("GPU (CUDA) aktif mi?:", torch.cuda.is_available())


Colab ortamında mıyız?: True
GPU (CUDA) aktif mi?: False


In [3]:
!pip install -q torch_geometric


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 42.5 MB/s eta 0:00:00


In [4]:
import torch
from torch_geometric.data import Data


def generate_scene_graph():
    num_nodes = torch.randint(4, 9, (1,)).item()

    object_types = torch.empty(num_nodes, dtype=torch.float)
    object_types[0] = 0  # Dron
    object_types[1] = 1  # Ana Hedef
    if num_nodes > 2:
        object_types[2:] = torch.randint(2, 4, (num_nodes - 2,), dtype=torch.long).float()

    positions = torch.rand(num_nodes, 2) * 100
    speeds = torch.rand(num_nodes, 1) * 20
    threat_scores = torch.rand(num_nodes, 1)
    x = torch.cat((object_types.unsqueeze(1), positions, speeds, threat_scores), dim=1)

    distances = torch.cdist(positions, positions)
    edge_mask = (distances < 35) & ~torch.eye(num_nodes, dtype=torch.bool)
    edge_index = edge_mask.nonzero(as_tuple=False).t().contiguous().to(torch.long)

    drone_to_obstacle = (distances[0, object_types == 3] < 25).any()
    target_to_civilian = (distances[1, object_types == 2] < 25).any()
    if drone_to_obstacle:
        label = 2  # Kaçın
    elif target_to_civilian:
        label = 1  # Gözlemle
    else:
        label = 0  # Güvenli Yaklaş

    y = torch.tensor([label], dtype=torch.long)
    return Data(x=x, edge_index=edge_index, y=y)


scene_graph = generate_scene_graph()
print(scene_graph)
print("x shape:", scene_graph.x.shape)
print("edge_index shape:", scene_graph.edge_index.shape)
print("y:", scene_graph.y)


Data(x=[4, 5], edge_index=[2, 4], y=[1])
x shape: torch.Size([4, 5])
edge_index shape: torch.Size([2, 4])
y: tensor([1])


In [5]:
from torch_geometric.loader import DataLoader

train_dataset = [generate_scene_graph() for _ in range(1000)]
test_dataset = [generate_scene_graph() for _ in range(200)]

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Eğitim grafı sayısı: {len(train_dataset)}")
print(f"Test grafı sayısı: {len(test_dataset)}")
print(f"Eğitim batch sayısı: {len(train_loader)}")
print(f"Test batch sayısı: {len(test_loader)}")


Eğitim grafı sayısı: 1000
Test grafı sayısı: 200
Eğitim batch sayısı: 32
Test batch sayısı: 7


In [6]:
import torch.nn as nn
from torch_geometric.nn import GCNConv, global_mean_pool


class TacticalDroneGNN(nn.Module):
    def __init__(self, input_dim=5, hidden_dim=32, num_classes=3):
        super().__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, data):
        x = self.conv1(data.x, data.edge_index)
        x = torch.relu(x)
        x = self.conv2(x, data.edge_index)
        x = torch.relu(x)
        x = global_mean_pool(x, data.batch)
        return self.classifier(x)


model = TacticalDroneGNN()
batch = next(iter(train_loader))
logits = model(batch)
print("Model çıktısı boyutu:", logits.shape)


Model çıktısı boyutu: torch.Size([32, 3])


In [ ]:
model = TacticalDroneGNN()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
num_epochs = 30

for epoch in range(1, num_epochs + 1):
    model.train()
    total_loss = 0.0

    for data in train_loader:
        optimizer.zero_grad()
        outputs = model(data)
        loss = criterion(outputs, data.y.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if epoch % 5 == 0:
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for data in test_loader:
                outputs = model(data)
                predictions = outputs.argmax(dim=1)
                labels = data.y.view(-1)
                correct += (predictions == labels).sum().item()
                total += labels.size(0)

        average_loss = total_loss / len(train_loader)
        accuracy = correct / total
        print(
            f"Epoch [{epoch}/{num_epochs}], "
            f"Loss: {average_loss:.4f}, "
            f"Test Accuracy: {accuracy:.4f}"
        )

torch.save(model.state_dict(), "tactical_gnn.pth")
print("Model tactical_gnn.pth olarak kaydedildi.")


Epoch [5/30], Loss: 0.9954, Test Accuracy: 0.5100
